# c-GC Full Analysis

This notebook runs the current compact effectome workflow end to end for `c-GC` using the corrected signed c-GC refit and prospective temporal role clustering.
Stages are checkpointed under `outputs/notebook_runs/<RUN_ID>/cgc/`, so rerunning the cells resumes completed work and only recomputes stages listed in `FORCE_STAGES`.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]

from effectome.community import CommunityConfig  # noqa: E402
from effectome.connectivity import ConnectivityConfig  # noqa: E402
from effectome.dynamics import (  # noqa: E402
    GraphStateConfig,
    ProbabilisticStateConfig,
    TransitionConfig,
)
from effectome.manifold import ManifoldConfig  # noqa: E402
from notebooks._shared import (  # noqa: E402
    PROJECT_ROOT,
    completed_stage_records,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_candidate_drivers,
    run_community,
    run_connectivity,
    run_feature_sensitivity,
    run_graph_states_and_transitions,
    run_linking,
    run_manifold,
    run_probabilistic_states,
)


In [ ]:
RUN_ID = 'demo-recording'
METHOD = 'cgc'
CONNECTIVITY_NAME = 'cgc'

OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'notebook_runs'
WINDOWS_PATH = PROJECT_ROOT / 'outputs' / 'artifacts' / 'windows.pkl'
RECORDING_PATH = PROJECT_ROOT / 'outputs' / 'artifacts' / 'recording.pkl'

FORCE_STAGES: set[str] = set()
CHUNK_SIZE = 16
BEHAVIOR_KEY = 'continuous'
TOP_K = 3
DOSE_SCALES = [-0.5, 0.5]

CONNECTIVITY_CFG = ConnectivityConfig(
    name=CONNECTIVITY_NAME,
    max_lag=2,
    alpha=0.05,
    ridge=1.0,
    extra={
        'support_test': 'analytic',
        'n_perm': 0,
        'n_lags': 2,
        'beta': 0.01,
        'lag_aggregation': 'sum',
        'simulation': True,
        'temporal': True,
        'seed': 42,
    },
)
GRAPH_CFG = GraphStateConfig(n_states=3, metric='causal_kernel', seed=42)
TRANSITION_CFG = TransitionConfig(n_null=500, heldout_fraction=0.3, seed=42)
PROB_CFG = ProbabilisticStateConfig(n_states=3, n_components=8, n_iter=50, seed=42)
COMMUNITY_CFG = CommunityConfig(
    name='temporal',
    symmetrize=False,
    use_absolute=False,
    seed=42,
    extra={
        'n_communities': 3,
        'mode': 'prospective',
        'temporal_penalty': 1.5,
        'n_runs': 5,
        'max_iter': 20,
        'init_window_count': 4,
        'online_learning_rate': 0.25,
    },
)
MANIFOLD_CFG = ManifoldConfig(
    name='classical',
    method='pca',
    n_dims=3,
    behavior_key=BEHAVIOR_KEY,
    target_length=15,
    cross_fit=True,
    n_folds=5,
    seed=42,
)
LINKING_CFG = {
    'behavior_keys': [BEHAVIOR_KEY],
    'n_null': 1000,
    'n_bins': 5,
    'max_lag': 10,
    'positive_lag': 1,
    'n_folds': 5,
    'embargo': 4,
    'lag_extension': 0,
    'preprocessing_past_support': 0,
    'preprocessing_future_support': 0,
    'allow_global_preprocessing': False,
    'group_by': 'recording',
    'null_kind': 'circular_shift',
    'block_length': 8,
    'seed': 42,
}

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title='c-GC full analysis sweep',
    expected_stages=[
        'windows_input',
        'connectivity',
        'graph_states',
        'transitions',
        'probabilistic_states',
        'community',
        'recording_input',
        'manifold',
        'linking',
        'candidate_drivers',
        'surrogate_model',
        'feature_sensitivity',
    ],
    force=bool(FORCE_STAGES),
)


In [ ]:
connectivity = run_connectivity(
    run,
    windows_path=WINDOWS_PATH,
    connectivity_cfg=CONNECTIVITY_CFG,
    chunk_size=CHUNK_SIZE,
    force='connectivity' in FORCE_STAGES,
)
states, transitions = run_graph_states_and_transitions(
    run,
    graph_cfg=GRAPH_CFG,
    transition_cfg=TRANSITION_CFG,
    force=bool({'graph_states', 'transitions'} & FORCE_STAGES),
)
probabilistic = run_probabilistic_states(
    run,
    probabilistic_cfg=PROB_CFG,
    force='probabilistic_states' in FORCE_STAGES,
)
community = run_community(
    run,
    community_cfg=COMMUNITY_CFG,
    force='community' in FORCE_STAGES,
)
manifold = run_manifold(
    run,
    recording_path=RECORDING_PATH,
    manifold_cfg=MANIFOLD_CFG,
    linking_cfg=LINKING_CFG,
    force='manifold' in FORCE_STAGES,
)
linking = run_linking(
    run,
    recording_path=RECORDING_PATH,
    linking_cfg=LINKING_CFG,
    force='linking' in FORCE_STAGES,
)
print_stage_status(run)


In [ ]:
candidate_drivers = run_candidate_drivers(
    run,
    recording_path=RECORDING_PATH,
    behavior_key=BEHAVIOR_KEY,
    lag=LINKING_CFG['positive_lag'],
    n_folds=LINKING_CFG['n_folds'],
    embargo=LINKING_CFG['embargo'],
    seed=LINKING_CFG['seed'],
    force='candidate_drivers' in FORCE_STAGES,
)
sensitivity = run_feature_sensitivity(
    run,
    recording_path=RECORDING_PATH,
    behavior_key=BEHAVIOR_KEY,
    lag=LINKING_CFG['positive_lag'],
    n_folds=LINKING_CFG['n_folds'],
    embargo=LINKING_CFG['embargo'],
    dose_scales=DOSE_SCALES,
    top_k=TOP_K,
    seed=LINKING_CFG['seed'],
    force=bool({'surrogate_model', 'feature_sensitivity', 'virtual_perturbation'} & FORCE_STAGES),
)
n_candidates = sum(score.is_candidate for score in candidate_drivers.scores)
print(f'Candidate drivers: {n_candidates}')
print(f'Feature sensitivity artifact: {type(sensitivity).__name__}')


In [ ]:
status_df = pd.DataFrame(completed_stage_records(run))
display(status_df[['stage', 'status', 'started_at', 'completed_at']])

summary = pd.Series(
    {
        'run_dir': str(run.run_dir),
        'connectivity_runtime_s': connectivity.diagnostics['runtime']['elapsed_seconds'],
        'transition_p_value': transitions.p_value,
        'transition_delta_ll': transitions.heldout_delta_log_likelihood,
        'hmm_log_likelihood': probabilistic.log_likelihood,
        'hmm_heldout_log_likelihood': probabilistic.heldout_log_likelihood,
        'duration_diagnostic_status': probabilistic.hsmm_candidate.status,
        'behavior_keys': list(linking['association'].keys()),
    }
)
display(summary)
